In [60]:
import os
import duckdb

duckdb_path = os.getenv("DUCKDB_PATH")
assert duckdb_path is not None, "Variable 'DUCKDB_PATH' must be defined"

con = duckdb.connect(duckdb_path)
con.sql("SELECT * FROM INFORMATION_SCHEMA.TABLES;").df()

,table_catalog,table_schema,table_name,table_type,self_referencing_column_name,reference_generation,user_defined_type_catalog,user_defined_type_schema,user_defined_type_name,is_insertable_into,is_typed,commit_action,TABLE_COMMENT
0,legislative,main,deputados,BASE TABLE,None,None,None,None,None,YES,NO,None,None
1,legislative,main,proposicoesTemas,BASE TABLE,None,None,None,None,None,YES,NO,None,None
2,legislative,main,votacoesObjetos,BASE TABLE,None,None,None,None,None,YES,NO,None,None
3,legislative,main,votacoesVotos,BASE TABLE,None,None,None,None,None,YES,NO,None,None
4,legislative,main,deputies_propositions,VIEW,None,None,None,None,None,NO,NO,None,None


In [108]:
import numpy as np
import networkx as nx
from matplotlib import pyplot as plt
import statistics

# Old

In [62]:
deputies_votes_grouped_themes_query = open("../queries/deputies_votes_grouped_themes.sql").read()

def query_votes_by_grouped_themes(theme_codes: list[int]):
    return con.execute(deputies_votes_grouped_themes_query, [theme_codes]).df()

In [6]:
themes = [34]

votes = query_votes_by_grouped_themes(themes)

votes

,deputado_id,votacao_id,voto
0,d_220593,v_2345484-83,Sim
1,d_204379,v_2345484-83,Não
2,d_221328,v_2345484-83,Sim
3,d_204528,v_2345484-83,Não
4,d_121948,v_2345484-83,Não
...,...,...,...
16161,d_220536,v_2498090-47,Não
16162,d_160632,v_2498090-47,Sim
16163,d_220558,v_2498090-47,Não
16164,d_160592,v_2498090-47,Não


In [7]:
G = nx.from_pandas_edgelist(votes, source='deputado_id', target='votacao_id', edge_attr='voto')

## Creating attribute-filled deputies graph

In [52]:
deputies_attrs = con.execute("""
    SELECT DISTINCT
        CONCAT('d_', vv.deputado_id) as node_id,
        vv.deputado_nome as nome,
        vv.deputado_siglaPartido as partido,
        vv.deputado_siglaUf as uf
    FROM votacoesVotos vv
    INNER JOIN (
        SELECT
            deputado_id,
            MAX(dataHoraVoto) as ultima_votacao
        FROM votacoesVotos
        GROUP BY deputado_id
    ) ultimo on vv.deputado_id = ultimo.deputado_id
        AND vv.dataHoraVoto = ultimo.ultima_votacao
""").df()

In [53]:
deputies_attrs

,node_id,nome,partido,uf
0,d_220008,Eliza Virgínia,PP,PB
1,d_73806,Roseana Sarney,MDB,MA
2,d_213274,Ricardo Silva,PSD,SP
3,d_73788,Ricardo Barros,PP,PR
4,d_220661,Matheus Noronha,PL,CE
...,...,...,...,...
616,d_160508,Afonso Florence,PT,BA
617,d_204517,Zé Vitor,PL,MG
618,d_220692,Dal Barreto,UNIÃO,BA
619,d_204507,Carla Zambelli,PL,SP


In [57]:
attrs = deputies_attrs.set_index('node_id').to_dict('index')

In [37]:
nx.set_node_attributes(G, attrs)

In [38]:
nx.write_gexf(G, '../graphs/deputy_votes.gexf')

## Creating the projection

In [43]:
deputies_nodes, polls_nodes = nx.bipartite.sets(G)

In [45]:
def signed_weighted_projection(B, deputies):
    G_proj = nx.Graph()
    G_proj.add_nodes_from(deputies)

    for u, v in nx.bipartite.weighted_projected_graph(B, deputies).edges():
        common_votes = list(nx.common_neighbors(B, u, v))

        if not common_votes:
            continue

        agreeCount = 0
        disagreeCount = 0

        for vote in common_votes:
            vote_u = B[u][vote]['voto']
            vote_v = B[v][vote]['voto']

            if vote_u == vote_v:
                agreeCount += 1
            else:
                disagreeCount += 1

        total = agreeCount + disagreeCount
        weight = (agreeCount - disagreeCount) / total

        G_proj.add_edge(u, v, weight=weight)

    return G_proj

In [56]:
G_proj = signed_weighted_projection(G, deputies_nodes)

In [58]:
nx.set_node_attributes(G_proj, attrs)

In [59]:
nx.write_gexf(G_proj, '../graphs/deputy_projection.gexf')

# Create a projection for each theme (with enough polls)

## Steps
1. Select the code and name of each theme
2. Create the deputies-vote relation graph for each theme
3. Create the deputies agreement projections from the relation graphs

### Select themes code and name

In [80]:
themes = con.execute("""
    SELECT
        pt.codTema as code,
        pt.tema as theme,
        COALESCE(SUM(pp.polls_count), 0)::INT as polls_count
    FROM proposicoesTemas pt
    LEFT JOIN (
        SELECT
            vo.proposicao_id,
            COUNT(vo.idVotacao) as polls_count
        FROM votacoesObjetos vo
        JOIN votacoesVotos vv ON vv.idVotacao = vo.idVotacao
        GROUP BY vo.proposicao_id
    ) pp ON pp.proposicao_id = pt.proposicao_id
    GROUP BY 1, 2
    ORDER BY 3 DESC
""").df()

themes

,code,theme,polls_count
0,34,Administração Pública,16661
1,70,Finanças Públicas e Orçamento,11161
2,40,Economia,6431
3,51,Estrutura Fundiária,3946
4,54,"Energia, Recursos Hídricos e Minerais",3126
5,67,Direito e Defesa do Consumidor,3126
6,44,Direitos Humanos e Minorias,3126
7,42,Direito Civil e Processual Civil,2901
8,66,"Indústria, Comércio e Serviços",2370
9,56,Saúde,2011


### Select only voted themes

In [81]:
voted_themes = themes[themes['polls_count'] > 0]

voted_themes

,code,theme,polls_count
0,34,Administração Pública,16661
1,70,Finanças Públicas e Orçamento,11161
2,40,Economia,6431
3,51,Estrutura Fundiária,3946
4,54,"Energia, Recursos Hídricos e Minerais",3126
5,67,Direito e Defesa do Consumidor,3126
6,44,Direitos Humanos e Minorias,3126
7,42,Direito Civil e Processual Civil,2901
8,66,"Indústria, Comércio e Serviços",2370
9,56,Saúde,2011


### Create deputy-votes graph for each theme

#### Select deputies votes for each theme

In [93]:
def select_deputies_votes_per_theme(code_theme: int):
    deputies_votes = con.execute("""
        SELECT
          CONCAT('d_', vv.deputado_id) AS deputado_id,
          CONCAT('v_', vv.idVotacao) AS votacao_id,
          vv.voto
        FROM votacoesVotos vv
        WHERE vv.voto IN ('Sim', 'Não')
          AND vv.idVotacao IN (
            SELECT DISTINCT vo.idVotacao
            FROM votacoesObjetos vo
            JOIN proposicoesTemas pt ON vo.proposicao_id = pt.proposicao_id
            WHERE pt.codTema = ?
          )
    """, [code_theme]).df()

    return deputies_votes

#### Save attributes for graph filtering

In [ ]:
deputies_info = con.execute("""
    SELECT DISTINCT
        CONCAT('d_', vv.deputado_id) as node_id,
        vv.deputado_nome as nome,
        vv.deputado_siglaPartido as partido,
        vv.deputado_siglaUf as uf
    FROM votacoesVotos vv
    INNER JOIN (
        SELECT
            deputado_id,
            MAX(dataHoraVoto) as ultima_votacao
        FROM votacoesVotos
        GROUP BY deputado_id
    ) ultimo on vv.deputado_id = ultimo.deputado_id
        AND vv.dataHoraVoto = ultimo.ultima_votacao
""").df()

deputies_attrs = deputies_info.set_index('node_id').to_dict('index')

deputies_attrs

#### Create graph for each theme

In [102]:
for theme in voted_themes.itertuples():
    code_theme = theme.code
    
    deputies_votes = select_deputies_votes_per_theme(code_theme)

    G = nx.from_pandas_edgelist(deputies_votes, source='deputado_id', target='votacao_id', edge_attr='voto')

    nx.set_node_attributes(G, deputies_attrs)

    nx.write_gexf(G, f'../graphs/theme_votes/votes_{code_theme}.gexf')

#### Create normalized projection for each graph

In [115]:
def generate_normalized_projection(B: nx.classes.graph.Graph, deputies: set):
    votes_nodes = [n for n in B.nodes() if n not in deputies]

    deputy_idx = {d: i for i, d in enumerate(deputies)}
    vote_idx = {v: i for i, v in enumerate(votes_nodes)}

    vote_matrix = np.full((len(deputies), len(votes_nodes)), np.nan)

    for deputy_id in deputies:
        d_i = deputy_idx[deputy_id]
        for vote in B.neighbors(deputy_id):
            if vote in vote_idx:
                vote_matrix[d_i, vote_idx[vote]] = 1 if B[deputy_id][vote]['voto'] == 'Sim' else 0

    G_proj = nx.Graph()
    G_proj.add_nodes_from(deputies)

    for u, v in nx.bipartite.projected_graph(B, deputies).edges():
        row_u = vote_matrix[deputy_idx[u]]
        row_v = vote_matrix[deputy_idx[v]]

        common_mask = ~np.isnan(row_u) & np.isnan(row_v)

        if not np.any(common_mask):
            continue

        u_votes = row_u[common_mask]
        v_votes = row_v[common_mask]

        agree = np.sum(u_votes == v_votes)
        disagree = np.sum(u_votes != v_votes)
        total = agree + disagree

        G_proj.add_edge(u, v, weight=(agree - disagree) / total)

    return G_proj

In [121]:
code_theme = 34

G = nx.read_gexf(f'../graphs/theme_votes/votes_{code_theme}.gexf')

deputies_nodes, _ = nx.bipartite.sets(G)

G_proj = generate_normalized_projection(G, deputies_nodes)

nx.set_node_attributes(G_proj, deputies_attrs)

nx.write_gexf(G_proj, f'../graphs/theme_projections/projection_{code_theme}.gexf')